In [1]:
import optuna
import torch
import os
import numpy as np
from solver import Solver
from data.data_loader import get_loader
from utils.genutils import write_print, mkdir

# Import your project configuration
import argparse

# Create a temporary configuration object
class Config:
    def __init__(self):
        self.lr = 0.001
        self.momentum = 0.9
        self.batch_size = 32
        self.num_epochs = 220
        self.dataset = 'tomatod'
        self.new_size = 300
        self.model = 'SSD'
        self.weight_decay = 0.0005
        self.learning_sched = [160, 190]
        self.use_gpu = torch.cuda.is_available()
        self.model_save_path = "./weights"
        self.model_test_path = "./tests"
        self.model_eval_path = "./eval"
        self.means = (104, 117, 123)  # Default values for normalization
        self.mode = "train"
        self.tomatod_data_path = "./data/Datasets/Tomatod/"
        self.class_count = 4
        self.anchor_config = 'SSD-300'  # Default anchor configuration
        self.scale_initial = 0.1
        self.scale_min = 0.2
        self.scale_max = 1.05

# Create a global config object
config = Config()

In [2]:
class ConfigObject:
    """Converts a dictionary into an object with attributes."""
    def __init__(self, config_dict):
        for key, value in config_dict.items():
            setattr(self, key, value)

In [14]:
def objective(trial):
    """Objective function for Optuna hyperparameter optimization"""

    # Step 1: Define the hyperparameters to tune with Optuna
    lr = trial.suggest_categorical('lr', [0.0001, 0.00001])  # Discrete steps for LR
    momentum = trial.suggest_categorical('momentum', [0.8, 0.85, 0.9, 0.95, 0.99])  # Cleaner momentum values
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])  # Batch Size
    num_epochs = trial.suggest_int('num_epochs', 50, 150, step=10)  # Number of epochs

    # Step 2: Define a complete config dictionary (NO missing keys)
    config_dict = {
        # Dataset information
        'input_channels': 3,
        'class_count': 4,
        'dataset': 'tomatod',  # Default dataset, update dynamically for Optuna
        'new_size': 300,
        'means': (104, 117, 123),
        'anchor_config': 'SSD-300',
        'scale_initial': 0.1,
        'scale_min': 0.2,
        'scale_max': 1.05,

        # Training settings
        'lr': lr,  # Tuned by Optuna
        'momentum': momentum,  # Tuned by Optuna
        'weight_decay': 0.0005,
        'num_epochs': num_epochs,  # Tuned by Optuna
        'learning_sched': [
            int(num_epochs * 0.5),  # 60% of total epochs
            int(num_epochs * 0.8)   # 80% of total epochs
        ],
        'warmup_epoch': 0,
        'sched_gamma': 0.1,
        'batch_size': batch_size,  # Tuned by Optuna
        'batch_multiplier': 1,

        # Model architecture settings
        'model': 'SSD',
        'basenet': 'vgg16_reducedfc.pth',
        'resnet_model': '18',
        'densenet_model': '121',
        'resnext_model': '50_32x4d',
        'pretrained_model': None,  # Default to None, update dynamically if needed
        'coco_weights': None,

        # Loss settings
        'loss_config': 'multibox',
        'pos_neg_ratio': 3,

        # Miscellaneous settings
        'mode': 'train',
        'use_gpu': torch.cuda.is_available(),

        # Testing settings
        'max_per_image': 50,
        'score_threshold': 0.01,
        'nms_threshold': 0.5,
        'iou_threshold': 0.5,

        # Dataset paths
        'voc_config': '0712',
        'voc_data_path': '../../Datasets/PascalVOC/',
        'use_07_metric': True,
        'coco_year': '2017',
        'coco_data_path': '../../Datasets/Coco/',
        'tomatod_data_path': 'data/Datasets/Tomatod/',
        'ccrop_data_path': '../../Datasets/CCROP/',
        'camocrops_data_path': '../../Datasets/CamoCrops/',

        # Paths
        'model_save_path': './weights',
        'model_test_path': './tests',
        'model_eval_path': './eval',

        # Logging settings
        'loss_log_step': 1,
        'model_save_step': 5,
    }

    config_obj = ConfigObject(config_dict)  # Convert dictionary to object

    # Step 3: Define a unique version name for each trial
    version = f"optuna_lr{lr:.6f}_mom{momentum:.6f}_bs{batch_size}_epochs{num_epochs}"

    # Ensure logs directory exists
    log_dir = "./logs"
    if not os.path.exists(log_dir):
        os.makedirs(log_dir)

    # Ensure model save directory exists
    model_save_dir = os.path.join(config_dict['model_save_path'], version)
    if not os.path.exists(model_save_dir):
        os.makedirs(model_save_dir)

    # Define output file path for logs
    output_txt = os.path.join(log_dir, f"{version}.txt")

    config_obj = ConfigObject(config_dict)  # Convert dictionary to object
    data_loader = get_loader(config_obj)  # Load dataset with current config


    # Print current hyperparameter values before training
    print("\nStarting Optuna Trial with Hyperparameters:")
    for key, value in config_dict.items():
        print(f"  {key}: {value}")
        
    
    solver = Solver(version=version, data_loader=data_loader, config=config_dict, output_txt=output_txt)

    
    # Step 4: Train the model with these hyperparameters
    solver.train()

    # Step 5: Retrieve validation loss
    val_loss = get_best_validation_loss(version)
    return val_loss

In [4]:
def get_best_validation_loss(version):
    """Reads the best validation loss from the log file"""
    log_path = f'./logs/{version}.txt'
    if not os.path.exists(log_path):
        return float('inf')  # Return a high loss if no log is found
    
    best_loss = float('inf')
    with open(log_path, 'r') as f:
        for line in f:
            if "val_loss" in line:  # Assuming your logs contain val_loss
                loss = float(line.split("val_loss: ")[1])
                best_loss = min(best_loss, loss)
    
    return best_loss


In [16]:
# Create an Optuna study to minimize validation loss
study = optuna.create_study(direction='minimize')

# Run optimization for 20 trials
study.optimize(objective, n_trials=20)

[I 2025-02-21 19:33:33,855] A new study created in memory with name: no-name-31e1f25e-34ba-4ac6-8573-0c3b32e45fa4



Starting Optuna Trial with Hyperparameters:
  input_channels: 3
  class_count: 4
  dataset: tomatod
  new_size: 300
  means: (104, 117, 123)
  anchor_config: SSD-300
  scale_initial: 0.1
  scale_min: 0.2
  scale_max: 1.05
  lr: 0.0001
  momentum: 0.9
  weight_decay: 0.0005
  num_epochs: 100
  learning_sched: [50, 80]
  warmup_epoch: 0
  sched_gamma: 0.1
  batch_size: 16
  batch_multiplier: 1
  model: SSD
  basenet: vgg16_reducedfc.pth
  resnet_model: 18
  densenet_model: 121
  resnext_model: 50_32x4d
  pretrained_model: None
  coco_weights: None
  loss_config: multibox
  pos_neg_ratio: 3
  mode: train
  use_gpu: True
  max_per_image: 50
  score_threshold: 0.01
  nms_threshold: 0.5
  iou_threshold: 0.5
  voc_config: 0712
  voc_data_path: ../../Datasets/PascalVOC/
  use_07_metric: True
  coco_year: 2017
  coco_data_path: ../../Datasets/Coco/
  tomatod_data_path: data/Datasets/Tomatod/
  ccrop_data_path: ../../Datasets/CCROP/
  camocrops_data_path: ../../Datasets/CamoCrops/
  model_save_

100%|██████████████████████████████████████████████████████████████████████████████████| 13/13 [00:26<00:00,  2.01s/it]


Elapsed 0:00:26.135575/0:00:02.010429 -- 0:43:09.432359, Epoch [1/100], Iter [13/13], class_loss: 8.8815, loc_loss: 5.6783, loss: 14.5598


100%|██████████████████████████████████████████████████████████████████████████████████| 13/13 [00:24<00:00,  1.86s/it]


...

Elapsed 0:09:53.775102/0:00:02.076137 -- 0:35:07.278771, Epoch [22/100], Iter [13/13], class_loss: 3.7135, loc_loss: 2.6289, loss: 6.3423


100%|██████████████████████████████████████████████████████████████████████████████████| 13/13 [00:26<00:00,  2.07s/it]


Elapsed 0:10:20.727731/0:00:02.076012 -- 0:34:40.164502, Epoch [23/100], Iter [13/13], class_loss: 3.1497, loc_loss: 1.4617, loss: 4.6114


  0%|                                                                                           | 0/13 [00:07<?, ?it/s]
[W 2025-02-21 19:44:02,203] Trial 0 failed with parameters: {'lr': 0.0001, 'momentum': 0.9, 'batch_size': 16, 'num_epochs': 100} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "C:\Users\audrea\anaconda3\envs\thesis\lib\site-packages\optuna\study\_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\audrea\AppData\Local\Temp\ipykernel_13936\2183292809.py", line 112, in objective
    solver.train()
  File "C:\Users\audrea\Desktop\sk00L\CIVI\SSD FINAL\SSD-PyTorch\solver.py", line 315, in train
    for i, (images, targets) in enumerate(tqdm(self.data_loader)):
  File "C:\Users\audrea\anaconda3\envs\thesis\lib\site-packages\tqdm\std.py", line 1181, in __iter__
    for obj in iterable:
  File "C:\Users\audrea\anaconda3\envs\thesis\lib\site-packages\torch\utils\data\dataloader.py", line 630, 

KeyboardInterrupt: 

In [ ]:
print("Best hyperparameters:", study.best_params)
